In [1]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

print(project_root)

/Users/rinugeorge/personal_portfolio_projects/rag/kb_rag


In [3]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import src.config.logging_config 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from src.ingestion import load_document
from src.embeddings import generate_embedding
from src.config.rag_config import ChunkConfig,EmbeddingConfig
from src.ingestion import index_builder

In [43]:
documents = load_document.load_pdfs()
chunk_config = ChunkConfig()
chunks = load_document.documents_to_chunks(documents,chunk_config)


2026-07-14 22:24:21,790 | INFO | src.ingestion.load_document | Loading documents
2026-07-14 22:24:27,953 | INFO | src.ingestion.load_document | Loaded 587 documents
2026-07-14 22:24:27,954 | INFO | src.ingestion.load_document | first file name eisenstein-nlp-notes.pdf
2026-07-14 22:24:27,956 | INFO | src.ingestion.load_document | spliting documents into chunks


In [ ]:
MIN_CHUNK_LENGTH = 100  # characters

filtered_chunks = [
    chunk for chunk in chunks
    if len(chunk.text.strip()) >= MIN_CHUNK_LENGTH
]


In [45]:
emb_config = EmbeddingConfig()

embed_model = generate_embedding.get_embedding_model(emb_config)

2026-07-14 22:24:41,515 | INFO | src.embeddings.generate_embedding | initializing embedding model
2026-07-14 22:24:41,662 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 22:24:41,683 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-14 22:24:41,714 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 22:24:41,735 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-14 22:24:41,738 | INFO | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-small-

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-14 22:24:42,204 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 22:24:42,236 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 22:24:42,268 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 22:24:42,313 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 22:24:42,343 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 22:24:42,362 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b

In [46]:
from llama_index.core import VectorStoreIndex

index = index_builder.build_index(filtered_chunks,embed_model)

2026-07-14 22:24:48,452 | INFO | src.ingestion.index_builder | Building index from 1103 chunks
2026-07-14 22:25:07,689 | INFO | src.ingestion.index_builder | Index created successfully


In [47]:
query_engine = index.as_query_engine()
response = query_engine.query(
    "What is the main topic of this document?"
)

print(response)

ValueError: 
******
Could not load OpenAI model. If you intended to use OpenAI, please check your OPENAI_API_KEY.
Original error:
No API key found for OpenAI.
Please set either the OPENAI_API_KEY environment variable or openai.api_key prior to initialization.
API keys can be found or created at https://platform.openai.com/account/api-keys

******

In [48]:
embedding = embed_model.get_text_embedding(
    "This is a test sentence"
)

print(len(embedding))
print(embedding[:5])

384
[-0.07318741083145142, 0.014326169155538082, 0.015609504655003548, 0.0034427656792104244, 0.014792689122259617]


In [49]:
retriever = index.as_retriever(
    similarity_top_k=3
)

results = retriever.retrieve(
    "Attention"
)

print("Retrieved:", len(results))

for result in results:
    print("Score:", result.score)
    print(result.node.text[:200])
    print("----------------")

Retrieved: 3
Score: 0.6880515737307207
These columns are both the keys and the values in the attention mechanism.
At each step m in decoding, the attentional state is computed by executing a query,
which is equal to the state of the decode
----------------
Score: 0.6864019472610396
18.3. NEURAL MACHINE TRANSLATION 445
Output
activation α
Query ψα
Key Value
Figure 18.6: A general view of neural attention. The dotted box indicates that each αm→n
can be viewed as a gate on valuen.

----------------
Score: 0.6689957420850173
17.5. QUESTION ANSWERING AND MACHINE READING 427
The attention vector is computed as a softmax over a vector of bilinear products, and
the expected representation is computed by summing over attention
----------------


In [31]:
import random

for i in range(3):
    chunk = random.choice(chunks)
    print(chunk.text[:500])
    print("----------------")

2.3. DISCRIMINATIVE LEARNING 25
To test the quality of the approximation, we can manipulate the left-hand side by applying
the chain rule,
Pr(word = unﬁt, preﬁx = un-|y) = Pr(preﬁx = un-| word = unﬁt,y ) [2.31]
× Pr(word = unﬁt|y) [2.32]
But Pr(preﬁx = un-| word = unﬁt,y ) = 1, since un- is guaranteed to be the preﬁx for the
word unﬁt. Therefore,
Pr(word = unﬁt, preﬁx = un-|y) =1 × Pr(word = unﬁt|y) [2.33]
≫ Pr(preﬁx = un-|y) × Pr(word = unﬁt|y), [2.34]
because the probability of any given word 
----------------
But rather than designing these feature ex-
tractors by hand, a better approach is to learn them, using the magic of backpropagation.
This is the idea behind convolutional neural networks.
Following§ 3.2.4, deﬁne the base layer of a neural network as,
X(0) = Θ(x→z)[ew1,ew2,..., ewM ], [3.52]
whereewm is a column vector of zeros, with a1 at positionwm. The base layer has dimen-
sion X(0)∈ RKe×M, whereKe is the size of the word embeddings. To merge information
across adjacent wor